# Zolai Bible Pipeline v1
**Parallel Bible Dataset Builder + Language Pattern Extractor**

Parses USX files across translations, aligns verses, cleans text, extracts language patterns.

| Translation | Language | ISO |
|---|---|---|
| TDB77 | Zolai/Tedim | ctd |
| Tedim 1932 | Zolai/Tedim | ctd |
| King James | English | eng |
| Judson 1835 | Myanmar | mya |
| HCL06 | Hakha Chin | cnh |
| FCL | Falam Chin | cfm |

**Kaggle:** Upload `Chin-Bible` as a dataset, add it to the notebook. Toggle Internet ON.

In [20]:
#%% Cell 1: Environment Check & Dependencies
import os, sys, platform, json, hashlib, re, socket
import unicodedata
from collections import defaultdict, Counter
from pathlib import Path

print(f"Platform: {platform.platform()}")
print(f"Python:   {sys.version}")
print(f"CWD:      {os.getcwd()}")

# Internet check
def has_internet():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
        return True
    except OSError:
        return False

INTERNET = has_internet()
print(f"Internet: {'ON' if INTERNET else 'OFF'}")

if INTERNET:
    !pip install -q -U "datasets>=2.19.0" "lxml" "pandas>=2.0,<3"
    print("Install complete.")
else:
    print("CRITICAL: Internet is OFF. Enable Internet in Kaggle Settings.")

import pandas as pd
from lxml import etree

print("\nAll imports OK")

Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Python:   3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CWD:      /kaggle/working
Internet: ON
Install complete.

All imports OK


In [21]:
#%% Cell 2: Config - Paths & Translation Registry
IS_KAGGLE = Path("/kaggle").exists()
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")

REPO_ROOT = Path(os.getcwd()).resolve()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()

# Bible data root - Kaggle dataset mount or local
if IS_KAGGLE:
    BIBLE_ROOT = None
    # 1) Try known Kaggle mount paths
    for candidate in [
        Path("/kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible"),
        Path("/kaggle/input/datasets/peterpausianlian/Chin-Bible"),
        Path("/kaggle/input/Chin-Bible"),
    ]:
        if candidate.exists():
            BIBLE_ROOT = candidate
            break
    # 2) Fallback: scan ALL of /kaggle/input for Chin-Bible directory
    if BIBLE_ROOT is None:
        print("Scanning /kaggle/input for Bible data...")
        for root, dirs, files in os.walk("/kaggle/input"):
            # Skip hidden/system dirs
            if Path(root).name.startswith("."):
                continue
            # Look for a directory named Chin-Bible
            if Path(root).name == "Chin-Bible":
                BIBLE_ROOT = Path(root)
                break
    # 3) Last resort: scan for any directory containing .usx files
    if BIBLE_ROOT is None:
        print("Scanning /kaggle/input for .usx files...")
        usx_dirs = set()
        for root, dirs, files in os.walk("/kaggle/input"):
            if any(f.endswith(".usx") for f in files):
                # Go up until we find a dir with multiple translation subdirs
                # (i.e. the Chin-Bible root, not a single translation subdir)
                usx_dirs.add(Path(root))
        # Find the common parent that contains the most .usx-containing subdirs
        if usx_dirs:
            # Collect all parent dirs
            parents = [p.parent for p in usx_dirs]
            # Find the shallowest parent that contains all USX dirs
            for depth in range(10):
                candidate = list(usx_dirs)[0]
                for _ in range(depth):
                    candidate = candidate.parent
                if candidate == Path("/"):
                    break
                # Check if this candidate contains all USX dirs
                if all(str(u).startswith(str(candidate)) for u in usx_dirs):
                    BIBLE_ROOT = candidate
                    break
    if BIBLE_ROOT is None:
        # Print what we found for debugging
        print("\nContents of /kaggle/input:")
        for item in sorted(Path("/kaggle/input").iterdir()):
            print(f"  {item}")
        raise FileNotFoundError(
            "Bible USX data not found. Add the Chin-Bible dataset in Kaggle Input."
        )
    OUTPUT_DIR = WORK_DIR / "zolai_bible_pipeline_out"
else:
    BIBLE_ROOT = Path(os.environ.get("ZOLOAI_BIBLE_ROOT", str(REPO_ROOT / "resources" / "Chin-Bible"))).resolve()
    OUTPUT_DIR = REPO_ROOT / "zolai_bible_pipeline_out"

OUTPUT_DIR.mkdir(exist_ok=True)

print(f"BIBLE_ROOT: {BIBLE_ROOT}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")

# Define all Bible translations to process
TRANSLATIONS = {
    "tdb77": {
        "name": "TDB77 (Tedim 1977)",
        "path": BIBLE_ROOT / "TDB77" / "USX_1",
        "language": "zolai",
        "iso": "ctd",
        "year": 1977,
        "is_target": True,
    },
    "tedim1932": {
        "name": "Tedim (Chin) Bible 1932",
        "path": BIBLE_ROOT / "Tedim (Chin) Bible)" / "USX_1",
        "language": "zolai",
        "iso": "ctd",
        "year": 1932,
        "is_target": True,
    },
    "kjv": {
        "name": "King James Version",
        "path": BIBLE_ROOT / "King James Version" / "USX_1",
        "language": "english",
        "iso": "eng",
        "year": 1611,
        "is_target": False,
    },
    "judson": {
        "name": "Judson Bible (Myanmar)",
        "path": BIBLE_ROOT / "Judson Bible" / "USX_1",
        "language": "myanmar",
        "iso": "mya",
        "year": 1835,
        "is_target": False,
    },
    "hcl06": {
        "name": "Hakha Chin (HCL06)",
        "path": BIBLE_ROOT / "HCL06" / "USX_1",
        "language": "hakha_chin",
        "iso": "cnh",
        "year": 2006,
        "is_target": False,
    },
    "fcl": {
        "name": "Falam Chin (FCL)",
        "path": BIBLE_ROOT / "FCL" / "USX_1",
        "language": "falam_chin",
        "iso": "cfm",
        "year": 0,
        "is_target": False,
    },
}

# Verify which translations exist
for key, t in TRANSLATIONS.items():
    exists = t["path"].exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {t['name']} -> {t['path']}")

Environment: Kaggle
BIBLE_ROOT: /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible
OUTPUT_DIR:  /kaggle/working/zolai_bible_pipeline_out
  [OK] TDB77 (Tedim 1977) -> /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible/TDB77/USX_1
  [OK] Tedim (Chin) Bible 1932 -> /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible/Tedim (Chin) Bible)/USX_1
  [OK] King James Version -> /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible/King James Version/USX_1
  [OK] Judson Bible (Myanmar) -> /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible/Judson Bible/USX_1
  [OK] Hakha Chin (HCL06) -> /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible/HCL06/USX_1
  [OK] Falam Chin (FCL) -> /kaggle/input/datasets/peterpausianlian/bible-datasets/Chin-Bible/FCL/USX_1


In [ ]:
#%% Path Checker — Show all available inputs
print("=" * 60)
print("KAGGLE INPUT PATH CHECKER")
print("=" * 60)

base = Path("/kaggle/input/datasets/peterpausianlian")
alt_base = Path("/kaggle/input")

print(f"\nBase path: /kaggle/input/datasets/peterpausianlian/")
if base.exists():
    print(f"  EXISTS: {base}")
    for item in sorted(base.iterdir()):
        if item.is_dir():
            subitems = list(item.iterdir())[:5]
            subnames = [p.name for p in subitems]
            more = f" (+{len(list(item.iterdir()))-5} more)" if len(list(item.iterdir())) > 5 else ""
            print(f"  /- {item.name}/")
            for s in subnames:
                print(f"     |- {s}")
            if more:
                print(f"     {more}")
else:
    print(f"  NOT FOUND: {base}")

print(f"\nAlt base: /kaggle/input/")
if alt_base.exists():
    for item in sorted(alt_base.iterdir()):
        if item.is_dir() and item.name not in (".", ".."):
            print(f"  /- {item.name}/")

print(f"\nWorking dir: /kaggle/working/")
working = Path("/kaggle/working")
if working.exists():
    for item in sorted(working.iterdir()):
        print(f"  |- {item.name}")

print("\n" + "=" * 60)

In [22]:
#%% Cell 3: USX Parser — Extract verses with book/chapter/verse keys

# Standard book order for alignment
BOOK_ORDER = [
    "GEN","EXO","LEV","NUM","DEU","JOS","JDG","RUT","1SA","2SA",
    "1KI","2KI","1CH","2CH","EZR","NEH","EST","JOB","PSA","PRO",
    "ECC","SNG","ISA","JER","LAM","EZK","DAN","HOS","JOL","AMO",
    "OBA","JON","MIC","NAM","HAB","ZEP","HAG","ZEC","MAL",
    "MAT","MRK","LUK","JHN","ACT","ROM","1CO","2CO","GAL","EPH",
    "PHP","COL","1TH","2TH","1TI","2TI","TIT","PHM","HEB","JAM",
    "1PE","2PE","1JN","2JN","3JN","JUD","REV"
]
BOOK_INDEX = {b: i for i, b in enumerate(BOOK_ORDER)}


def parse_usx_verse_id(filepath):
    """
    Parse a USX file and return {"book.CH.verse": text} dict.
    Handles: <verse number="1" style="v" />, <verse number="17-18" style="v" />
    Strips: <note>, <char>, <ref> tags and footnotes.
    """
    verses = {}
    book_code = Path(filepath).stem.upper()  # e.g. GEN

    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f:
            content = f.read()
    except Exception as e:
        print(f"  ERROR reading {filepath}: {e}")
        return verses

    # Remove all <note>...</note> blocks (footnotes/cross-refs)
    content = re.sub(r'<note[^>]*>.*?</note>', '', content, flags=re.DOTALL)
    # Remove <char> tags but keep their text
    content = re.sub(r'<char[^>]*>', '', content)
    content = re.sub(r'</char>', '', content)
    # Remove <ref> tags
    content = re.sub(r'<ref[^>]*/>', '', content)

    current_chapter = 0

    # Find chapter markers
    for match in re.finditer(r'<chapter number="(\d+)"', content):
        current_chapter = int(match.group(1))

    # Better approach: iterate through chapter and verse markers
    # Split by chapter markers first
    chapter_splits = re.split(r'<chapter number="(\d+)"[^>]*/>', content)
    # chapter_splits = [pre-chapter, ch_num, content, ch_num, content, ...]

    current_ch = 0
    for i, part in enumerate(chapter_splits):
        if part.strip().isdigit():
            current_ch = int(part.strip())
            continue
        if current_ch == 0:
            continue

        # Now extract verses from this chapter's content
        verse_pattern = r'<verse number="([^"]+)"[^>]*/>(.*?)(?=<verse|</para|<chapter|$)'
        for vmatch in re.finditer(verse_pattern, part, re.DOTALL):
            vnum = vmatch.group(1)  # e.g. "1" or "17-18"
            vtext = vmatch.group(2)
            # Strip any remaining XML tags
            vtext = re.sub(r'<[^>]+>', '', vtext)
            vtext = vtext.strip()
            if not vtext or len(vtext) < 3:
                continue
            # Handle verse ranges like "17-18": store under each verse number
            if '-' in vnum and not vnum.startswith('-'):
                parts = vnum.split('-')
                if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
                    for vn in range(int(parts[0]), int(parts[1]) + 1):
                        key = f"{book_code}.{current_ch}.{vn}"
                        verses[key] = vtext
                    continue
            key = f"{book_code}.{current_ch}.{vnum}"
            verses[key] = vtext

    return verses


def parse_all_translations():
    """Parse all available USX files and return {trans_key: {verse_key: text}}"""
    all_data = {}
    for tkey, tinfo in TRANSLATIONS.items():
        usx_dir = tinfo["path"]
        if not usx_dir.exists():
            print(f"SKIP {tinfo['name']}: directory not found")
            continue

        trans_verses = {}
        usx_files = sorted([f for f in usx_dir.iterdir() if f.suffix.lower() == '.usx'])
        print(f"Parsing {tinfo['name']}: {len(usx_files)} USX files...")
        for f in usx_files:
            file_verses = parse_usx_verse_id(f)
            trans_verses.update(file_verses)

        print(f"  -> {len(trans_verses):,} verses extracted")
        all_data[tkey] = trans_verses

    return all_data


raw_bibles = parse_all_translations()
print(f"\nTotal translations parsed: {len(raw_bibles)}")
for k, v in raw_bibles.items():
    print(f"  {k}: {len(v):,} verses")

Parsing TDB77 (Tedim 1977): 66 USX files...
  -> 30,793 verses extracted
Parsing Tedim (Chin) Bible 1932: 66 USX files...
  -> 30,808 verses extracted
Parsing King James Version: 66 USX files...
  -> 31,102 verses extracted
Parsing Judson Bible (Myanmar): 66 USX files...
  -> 31,075 verses extracted
Parsing Hakha Chin (HCL06): 66 USX files...
  -> 28,201 verses extracted
Parsing Falam Chin (FCL): 66 USX files...
  -> 29,661 verses extracted

Total translations parsed: 6
  tdb77: 30,793 verses
  tedim1932: 30,808 verses
  kjv: 31,102 verses
  judson: 31,075 verses
  hcl06: 28,201 verses
  fcl: 29,661 verses


In [23]:
#%% Cell 4: Parallel Alignment — Build verse-aligned parallel corpus

def build_parallel_corpus(all_data):
    """
    Align verses across all translations by verse key.
    Returns list of dicts: {"verse_id", "book", "chapter", "verse", "zolai", "english", ...}
    """
    # Collect all unique verse keys
    all_keys = set()
    for trans_verses in all_data.values():
        all_keys.update(trans_verses.keys())

    # Sort keys by book order, then chapter, then verse
    def sort_key(vk):
        parts = vk.split('.')
        book_idx = BOOK_INDEX.get(parts[0], 999)
        ch = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
        vs = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
        return (book_idx, ch, vs)

    sorted_keys = sorted(all_keys, key=sort_key)
    print(f"Total unique verse keys: {len(sorted_keys):,}")

    parallel = []
    for vk in sorted_keys:
        parts = vk.split('.')
        entry = {
            "verse_id": vk,
            "book": parts[0],
            "chapter": int(parts[1]) if len(parts) > 1 else 0,
            "verse": parts[2] if len(parts) > 2 else "0",
        }
        # Add text for each translation
        for tkey in TRANSLATIONS:
            entry[tkey] = all_data.get(tkey, {}).get(vk, None)
        parallel.append(entry)

    return parallel


parallel_corpus = build_parallel_corpus(raw_bibles)
print(f"Parallel corpus: {len(parallel_corpus):,} verse rows")

# Stats: how many translations per verse
trans_keys = list(TRANSLATIONS.keys())
coverage = Counter()
for row in parallel_corpus:
    count = sum(1 for k in trans_keys if row.get(k) is not None)
    coverage[count] += 1
print("\nVerse coverage (num translations present):")
for c in sorted(coverage):
    print(f"  {c} translations: {coverage[c]:,} verses")

# Show sample
print("\nSample (Genesis 1:1):")
for row in parallel_corpus:
    if row["verse_id"] == "GEN.1.1":
        for k in trans_keys:
            if row.get(k):
                print(f"  {k}: {row[k]}")
        break

Total unique verse keys: 31,105
Parallel corpus: 31,105 verse rows

Verse coverage (num translations present):
  1 translations: 1 verses
  3 translations: 31 verses
  4 translations: 419 verses
  5 translations: 4,054 verses
  6 translations: 26,600 verses

Sample (Genesis 1:1):
  tdb77: A kipat cil-in Pasian in vantung leh leitung a piangsak hi.
  tedim1932: A kipat cilin Pasian in vantung leh leitung a piangsak hi.
  kjv: In the beginning God created the heaven and the earth.
  judson: အစအဦး၌ ဘုရားသခင်သည် ကောင်းကင်နှင့် မြေကြီးကို ဖန်ဆင်းတော်မူ၏။
  fcl: A hmaisabik ah Pathian in lei le van tla a seemsuah.


In [24]:
#%% Cell 5: Text Cleaning Pipeline

def clean_text(text, language="unknown"):
    """Clean and normalize text for AI training."""
    if not text:
        return None

    # Unicode NFKC normalization
    text = unicodedata.normalize('NFKC', text)

    # Strip XML/HTML remnants (should be gone, but safety net)
    text = re.sub(r'<[^>]+>', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove trailing footnote markers like +, †, ‡, *
    text = re.sub(r'[+†‡*]+$', '', text).strip()

    # Normalize curly quotes to straight (English)
    if language in ("english", "unknown"):
        text = text.replace('\u201c', '"').replace('\u201d', '"')
        text = text.replace('\u2018', "'").replace('\u2019', "'")
        text = text.replace('\u2013', '-').replace('\u2014', '-')

    # Minimum length check
    if len(text) < 3:
        return None

    return text


def clean_parallel_corpus(parallel):
    """Clean all text fields in parallel corpus."""
    cleaned = []
    stats = {"total": 0, "cleaned": 0, "dropped": 0}

    for row in parallel:
        stats["total"] += 1
        has_any_text = False
        new_row = {"verse_id": row["verse_id"], "book": row["book"],
                   "chapter": row["chapter"], "verse": row["verse"]}

        for tkey, tinfo in TRANSLATIONS.items():
            raw = row.get(tkey)
            c = clean_text(raw, tinfo["language"])
            new_row[tkey] = c
            if c:
                has_any_text = True

        if has_any_text:
            cleaned.append(new_row)
            stats["cleaned"] += 1
        else:
            stats["dropped"] += 1

    print(f"Cleaning: {stats['total']:,} -> {stats['cleaned']:,} kept, {stats['dropped']:,} dropped")
    return cleaned


cleaned_corpus = clean_parallel_corpus(parallel_corpus)
print(f"Cleaned parallel corpus: {len(cleaned_corpus):,} verses")

Cleaning: 31,105 -> 31,105 kept, 0 dropped
Cleaned parallel corpus: 31,105 verses


In [25]:
#%% Cell 6: Deduplication

def dedupe_corpus(corpus):
    """Remove duplicate verses (same verse_id with identical text)."""
    seen = {}
    deduped = []
    dupes = 0

    for row in corpus:
        # Build a hash from all text fields
        text_parts = "|".join(str(row.get(k, "")) for k in trans_keys)
        h = hashlib.md5(text_parts.encode('utf-8')).hexdigest()
        if h not in seen:
            seen[h] = row["verse_id"]
            deduped.append(row)
        else:
            dupes += 1

    print(f"Dedup: {len(corpus):,} -> {len(deduped):,} ({dupes:,} duplicates removed)")
    return deduped


deduped_corpus = dedupe_corpus(cleaned_corpus)

Dedup: 31,105 -> 31,085 (20 duplicates removed)


In [26]:
#%% Cell 7: Output — Memory-Optimized Formats

# 7a. Parallel JSONL (full aligned corpus)
parallel_jsonl = OUTPUT_DIR / "bible_parallel.jsonl"
with open(parallel_jsonl, 'w', encoding='utf-8') as f:
    for row in deduped_corpus:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f"Saved parallel JSONL: {parallel_jsonl} ({len(deduped_corpus):,} lines)")


# 7b. Per-language training JSONL (flat text for each language)
for tkey, tinfo in TRANSLATIONS.items():
    lang_lines = []
    for row in deduped_corpus:
        text = row.get(tkey)
        if text:
            lang_lines.append({
                "text": text,
                "language": tinfo["language"],
                "iso": tinfo["iso"],
                "source": f"bible_{tkey}",
                "verse_id": row["verse_id"],
                "book": row["book"],
                "chapter": row["chapter"],
                "verse": row["verse"],
            })

    out_path = OUTPUT_DIR / f"bible_{tkey}.jsonl"
    with open(out_path, 'w', encoding='utf-8') as f:
        for line in lang_lines:
            f.write(json.dumps(line, ensure_ascii=False) + '\n')
    print(f"  {tkey}: {len(lang_lines):,} verses -> {out_path.name}")


# 7c. Zolai combined training set (merge all Zolai editions, dedupe by text)
zolai_texts = set()
zolai_lines = []
zolai_keys = [k for k, v in TRANSLATIONS.items() if v["is_target"]]
for row in deduped_corpus:
    for zk in zolai_keys:
        text = row.get(zk)
        if text and text not in zolai_texts:
            zolai_texts.add(text)
            zolai_lines.append({
                "text": text,
                "language": "zolai",
                "dialect": "tedim",
                "source": f"bible_{zk}",
                "verse_id": row["verse_id"],
            })

zolai_out = OUTPUT_DIR / "bible_zolai_combined.jsonl"
with open(zolai_out, 'w', encoding='utf-8') as f:
    for line in zolai_lines:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')
print(f"\nZolai combined: {len(zolai_lines):,} unique verses -> {zolai_out.name}")

# 7d. Per-book parallel JSONL + per-book Zolai training JSONL
per_book_dir = OUTPUT_DIR / "per_book"
per_book_dir.mkdir(exist_ok=True)
for book in BOOK_ORDER:
    book_rows = [r for r in deduped_corpus if r["book"] == book]
    if not book_rows:
        continue
    # Per-book parallel
    out = per_book_dir / f"bible_parallel_{book}.jsonl"
    with open(out, 'w', encoding='utf-8') as f:
        for row in book_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    # Per-book Zolai training
    book_zolai = []
    for row in book_rows:
        for zk in zolai_keys:
            text = row.get(zk)
            if text:
                book_zolai.append({"text": text, "verse_id": row["verse_id"], "book": book})
    if book_zolai:
        out = per_book_dir / f"bible_zolai_{book}.jsonl"
        with open(out, 'w', encoding='utf-8') as f:
            for line in book_zolai:
                f.write(json.dumps(line, ensure_ascii=False) + '\n')
print(f"Per-book files written to {per_book_dir}")


Saved parallel JSONL: /kaggle/working/zolai_bible_pipeline_out/bible_parallel.jsonl (31,085 lines)
  tdb77: 30,773 verses -> bible_tdb77.jsonl
  tedim1932: 30,788 verses -> bible_tedim1932.jsonl
  kjv: 31,082 verses -> bible_kjv.jsonl
  judson: 31,055 verses -> bible_judson.jsonl
  hcl06: 28,181 verses -> bible_hcl06.jsonl
  fcl: 29,656 verses -> bible_fcl.jsonl

Zolai combined: 60,085 unique verses -> bible_zolai_combined.jsonl
Per-book files written to /kaggle/working/zolai_bible_pipeline_out/per_book


In [27]:
#%% Cell 8: Language Pattern Extraction — Vocabulary, N-grams, Frequencies

def extract_word_freq(texts, top_n=50):
    """Extract word frequency from list of texts."""
    words = []
    for t in texts:
        words.extend(t.lower().split())
    return Counter(words).most_common(top_n)


def extract_ngrams(texts, n=2, top_n=30):
    """Extract n-gram frequency."""
    ngrams = []
    for t in texts:
        tokens = t.lower().split()
        for i in range(len(tokens) - n + 1):
            ngrams.append(tuple(tokens[i:i+n]))
    return Counter(ngrams).most_common(top_n)


patterns = {}
for tkey, tinfo in TRANSLATIONS.items():
    texts = [row[tkey] for row in deduped_corpus if row.get(tkey)]
    if not texts:
        continue

    vocab = set()
    for t in texts:
        vocab.update(t.lower().split())

    patterns[tkey] = {
        "language": tinfo["language"],
        "total_verses": len(texts),
        "total_tokens": sum(len(t.split()) for t in texts),
        "vocab_size": len(vocab),
        "top_words": extract_word_freq(texts, 30),
        "top_bigrams": extract_ngrams(texts, 2, 20),
        "top_trigrams": extract_ngrams(texts, 3, 15),
    }

    p = patterns[tkey]
    print(f"\n{'='*60}")
    print(f"{tinfo['name']} ({tinfo['language']})")
    print(f"  Verses: {p['total_verses']:,} | Tokens: {p['total_tokens']:,} | Vocab: {p['vocab_size']:,}")
    print(f"  Top 10 words: {', '.join(f'{w}({c})' for w,c in p['top_words'][:10])}")
    print(f"  Top 5 bigrams: {', '.join(f'{' '.join(ng)}({c})' for ng,c in p['top_bigrams'][:5])}")

# Save patterns
patterns_out = OUTPUT_DIR / "language_patterns.json"
with open(patterns_out, 'w', encoding='utf-8') as f:
    json.dump(patterns, f, ensure_ascii=False, indent=2)
print(f"\nSaved language patterns: {patterns_out}")


TDB77 (Tedim 1977) (zolai)
  Verses: 30,773 | Tokens: 838,848 | Vocab: 25,866
  Top 10 words: a(52119), hi.(30900), in(29986), uh(21967), ding(17560), na(17240), a,(13832), leh(13789), ka(13243), hong(12460)
  Top 5 bigrams: uh hi.(8180), ding hi.(4316), in a(4209), ding uh(3963), tua ciangin(3664)

Tedim (Chin) Bible 1932 (zolai)
  Verses: 30,788 | Tokens: 848,434 | Vocab: 28,807
  Top 10 words: a(59287), hi.(36636), in(23777), uh(22349), ding(19476), na(16862), a,(15780), leh(15531), hong(15486), ka(13378)
  Top 5 bigrams: uh hi.(9504), ding hi.(5071), ding uh(4399), topa }(4367), tua ciangin(4213)

King James Version (english)
  Verses: 31,082 | Tokens: 789,448 | Vocab: 27,435
  Top 10 words: the(63898), and(51303), of(34564), to(13550), that(12787), in(12503), he(10261), shall(9837), unto(8984), for(8800)
  Top 5 bigrams: of the(11516), and the(6259), in the(5028), the lord(4398), and he(2785)

Judson Bible (Myanmar) (myanmar)
  Verses: 31,055 | Tokens: 272,076 | Vocab: 138,432
  

In [28]:
#%% Cell 9: Parallel Alignment Patterns — Zolai↔English word alignment pairs

def extract_parallel_pairs(corpus, src_key, tgt_key, max_pairs=500):
    """
    Extract sentence-level parallel pairs for language learning.
    Returns list of {source, target, verse_id}.
    """
    pairs = []
    for row in corpus:
        src = row.get(src_key)
        tgt = row.get(tgt_key)
        if src and tgt and len(src) > 10 and len(tgt) > 10:
            pairs.append({
                "source": src,
                "target": tgt,
                "verse_id": row["verse_id"],
                "src_lang": TRANSLATIONS[src_key]["language"],
                "tgt_lang": TRANSLATIONS[tgt_key]["language"],
            })
    return pairs[:max_pairs]


# Build Zolai-English parallel pairs (primary training pair)
zolai_eng_pairs = []
for zk in zolai_keys:
    pairs = extract_parallel_pairs(deduped_corpus, zk, "kjv", max_pairs=99999)
    zolai_eng_pairs.extend(pairs)

# Dedupe by verse_id
seen_vids = set()
unique_pairs = []
for p in zolai_eng_pairs:
    if p["verse_id"] not in seen_vids:
        seen_vids.add(p["verse_id"])
        unique_pairs.append(p)

pairs_out = OUTPUT_DIR / "bible_zolai_english_pairs.jsonl"
with open(pairs_out, 'w', encoding='utf-8') as f:
    for p in unique_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f"Zolai↔English pairs: {len(unique_pairs):,} -> {pairs_out.name}")

# Show sample pairs
print("\nSample parallel pairs:")
for p in unique_pairs[:5]:
    print(f"  [{p['verse_id']}]")
    print(f"    Zolai:   {p['source'][:80]}")
    print(f"    English: {p['target'][:80]}")

Zolai↔English pairs: 31,053 -> bible_zolai_english_pairs.jsonl

Sample parallel pairs:
  [GEN.1.1]
    Zolai:   A kipat cil-in Pasian in vantung leh leitung a piangsak hi.
    English: In the beginning God created the heaven and the earth.
  [GEN.1.2]
    Zolai:   Leitung in lim leh meel nei loin a awngthawlpi ahi hi. A thuk tuipi tung tengah 
    English: And the earth was without form, and void; and darkness was upon the face of the 
  [GEN.1.3]
    Zolai:   Pasian in, “Khuavak om ta hen,” ci hi; tua ciangin khuavak om pah hi.
    English: And God said, Let there be light: and there was light.
  [GEN.1.4]
    Zolai:   Pasian in khuavak hoih hi ci-in mu hi; Pasian in khuamial panin khuavak khenkhia
    English: And God saw the light, that it was good: and God divided the light from the dark
  [GEN.1.5]
    Zolai:   Pasian in khuavak pen Sun ci a, khuamial pen Zan ci hi. Nitak hong bei-in, zings
    English: And God called the light Day, and the darkness he called Night. And the evenin

In [29]:
#%% Cell 10: HuggingFace Dataset Export (memory-optimized)
try:
    from datasets import Dataset, DatasetDict

    # Per-language datasets
    hf_datasets = {}
    for tkey, tinfo in TRANSLATIONS.items():
        rows = []
        for row in deduped_corpus:
            text = row.get(tkey)
            if text:
                rows.append({
                    "text": text,
                    "language": tinfo["language"],
                    "verse_id": row["verse_id"],
                    "book": row["book"],
                    "chapter": row["chapter"],
                })
        if rows:
            hf_datasets[tkey] = Dataset.from_list(rows)

    # Parallel dataset
    parallel_rows = []
    for row in deduped_corpus:
        entry = {"verse_id": row["verse_id"], "book": row["book"]}
        for k in trans_keys:
            entry[k] = row.get(k)
        parallel_rows.append(entry)
    hf_datasets["parallel"] = Dataset.from_list(parallel_rows)

    # Save all
    dd = DatasetDict(hf_datasets)
    hf_out = OUTPUT_DIR / "hf_bible_dataset"
    dd.save_to_disk(str(hf_out))
    print(f"Saved HuggingFace DatasetDict: {hf_out}")
    print(f"  Splits: {list(dd.keys())}")
    for k, ds in dd.items():
        print(f"    {k}: {len(ds):,} rows, {ds.features}")
except ImportError:
    print("datasets library not available, skipping HF export")

Saving the dataset (0/1 shards):   0%|          | 0/30773 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30788 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/31082 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/31055 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/28181 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/29656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/31085 [00:00<?, ? examples/s]

Saved HuggingFace DatasetDict: /kaggle/working/zolai_bible_pipeline_out/hf_bible_dataset
  Splits: ['tdb77', 'tedim1932', 'kjv', 'judson', 'hcl06', 'fcl', 'parallel']
    tdb77: 30,773 rows, {'text': Value('string'), 'language': Value('string'), 'verse_id': Value('string'), 'book': Value('string'), 'chapter': Value('int64')}
    tedim1932: 30,788 rows, {'text': Value('string'), 'language': Value('string'), 'verse_id': Value('string'), 'book': Value('string'), 'chapter': Value('int64')}
    kjv: 31,082 rows, {'text': Value('string'), 'language': Value('string'), 'verse_id': Value('string'), 'book': Value('string'), 'chapter': Value('int64')}
    judson: 31,055 rows, {'text': Value('string'), 'language': Value('string'), 'verse_id': Value('string'), 'book': Value('string'), 'chapter': Value('int64')}
    hcl06: 28,181 rows, {'text': Value('string'), 'language': Value('string'), 'verse_id': Value('string'), 'book': Value('string'), 'chapter': Value('int64')}
    fcl: 29,656 rows, {'text':

In [30]:
#%% Cell 11: Memory Stats & Summary Report
import os

print("=" * 60)
print("ZOLAI BIBLE PIPELINE — SUMMARY")
print("=" * 60)

total_size = 0
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"{'File':<45} {'Lines':>10} {'Size':>10}")
print("-" * 65)
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        total_size += size
        lines = "-"
        if f.suffix in (".jsonl",):
            with open(f, 'r', encoding='utf-8') as fh:
                lines = f"{sum(1 for _ in fh):,}"
        size_str = f"{size/1024:.1f}KB" if size < 1024*1024 else f"{size/1024/1024:.1f}MB"
        print(f"  {f.name:<43} {lines:>10} {size_str:>10}")

print("-" * 65)
print(f"  {'TOTAL':<43} {'':>10} {total_size/1024/1024:.1f}MB")

print(f"\nVerse alignment stats:")
print(f"  Total verses: {len(deduped_corpus):,}")
print(f"  Zolai unique verses: {len(zolai_lines):,}")
print(f"  Zolai↔English pairs: {len(unique_pairs):,}")

print(f"\nMemory optimization:")
print(f"  - JSONL streaming (no full RAM load needed)")
print(f"  - UTF-8 NFKC normalized")
print(f"  - Deduplicated by verse_id + text hash")
print(f"  - Separate per-language files for lazy loading")
print(f"  - HF DatasetDict with arrow format (columnar, memory-mapped)")


per_book_count = len(list((OUTPUT_DIR / "per_book").glob("*.jsonl")))
print(f"  Per-book files: {per_book_count}")
print("\nDone!")

ZOLAI BIBLE PIPELINE — SUMMARY

Output directory: /kaggle/working/zolai_bible_pipeline_out
File                                               Lines       Size
-----------------------------------------------------------------
  bible_fcl.jsonl                                 29,656      8.1MB
  bible_hcl06.jsonl                               28,181      7.8MB
  bible_judson.jsonl                              31,055     15.5MB
  bible_kjv.jsonl                                 31,082      8.1MB
  bible_parallel.jsonl                            31,085     35.7MB
  bible_tdb77.jsonl                               30,773      8.3MB
  bible_tedim1932.jsonl                           30,788      8.5MB
  bible_zolai_combined.jsonl                      60,085     14.2MB
  bible_zolai_english_pairs.jsonl                 31,053     11.0MB
  dataset_dict.json                                    -      0.1KB
  data-00000-of-00001.arrow                            -      5.3MB
  dataset_info.json        

In [31]:
#%% Cell 12: ZIP for Download + Kaggle Dataset Upload
import zipfile
import shutil

ZIP_NAME = "zolai_bible_dataset.zip"
ZIP_PATH = WORK_DIR / ZIP_NAME

print(f"Creating ZIP: {ZIP_PATH}")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(OUTPUT_DIR.rglob("*")):
        if fpath.is_file():
            arcname = str(fpath.relative_to(OUTPUT_DIR))
            zf.write(fpath, arcname)
            print(f"  + {arcname}")

zip_size = ZIP_PATH.stat().st_size
print(f"\nZIP created: {ZIP_NAME} ({zip_size/1024/1024:.1f} MB)")

if IS_KAGGLE:
    print("\n" + "=" * 60)
    print("KAGGLE DATASET UPLOAD INSTRUCTIONS")
    print("=" * 60)
    print("""
Option A — Download ZIP + Upload as New Dataset:
  1. Download zolai_bible_dataset.zip from Output panel
  2. Go to kaggle.com → Datasets → New Dataset
  3. Upload the ZIP, name it 'zolai-bible-dataset'
  4. Set License, make it Public/Private

Option B — Use Kaggle API in this notebook:
  1. Add your kaggle.json as a Kaggle Secret
  2. Run the upload code below

Option C — Save as Notebook Output Dataset:
  1. Save Version of this notebook
  2. The /kaggle/working/ output becomes a dataset
  3. Other notebooks can add it via Add Input
""")

    # Option B: Kaggle API upload (uncomment to use)
    UPLOAD_TO_KAGGLE = False  # Set True to upload
    if UPLOAD_TO_KAGGLE:
        !pip install -q kaggle 2>/dev/null
        import kaggle
        dataset_slug = "peterpausianlian/zolai-bible-dataset"
        print(f"Uploading to {dataset_slug}...")
        kaggle.api.dataset_create_version(
            dataset_slug,
            folder=str(OUTPUT_DIR),
            version_notes="Auto-generated by zolai-bible-pipeline-v1"
        )
        print("Upload complete!")

Creating ZIP: /kaggle/working/zolai_bible_dataset.zip
  + bible_fcl.jsonl
  + bible_hcl06.jsonl
  + bible_judson.jsonl
  + bible_kjv.jsonl
  + bible_parallel.jsonl
  + bible_tdb77.jsonl
  + bible_tedim1932.jsonl
  + bible_zolai_combined.jsonl
  + bible_zolai_english_pairs.jsonl
  + hf_bible_dataset/dataset_dict.json
  + hf_bible_dataset/fcl/data-00000-of-00001.arrow
  + hf_bible_dataset/fcl/dataset_info.json
  + hf_bible_dataset/fcl/state.json
  + hf_bible_dataset/hcl06/data-00000-of-00001.arrow
  + hf_bible_dataset/hcl06/dataset_info.json
  + hf_bible_dataset/hcl06/state.json
  + hf_bible_dataset/judson/data-00000-of-00001.arrow
  + hf_bible_dataset/judson/dataset_info.json
  + hf_bible_dataset/judson/state.json
  + hf_bible_dataset/kjv/data-00000-of-00001.arrow
  + hf_bible_dataset/kjv/dataset_info.json
  + hf_bible_dataset/kjv/state.json
  + hf_bible_dataset/parallel/data-00000-of-00001.arrow
  + hf_bible_dataset/parallel/dataset_info.json
  + hf_bible_dataset/parallel/state.json
  

# Next Steps & Reuse Guide

## Download Your Dataset
1. **ZIP file**: `zolai_bible_dataset.zip` is in `/kaggle/working/` — download from the Output panel (right sidebar)
2. **Save Version**: Click **Save Version** → output becomes a Kaggle dataset automatically
3. **Upload as Dataset**: Go to kaggle.com → Datasets → New → upload the ZIP

## Reuse in Other Notebooks
```python
# Add the dataset via Add Input, then:
BIBLE_DATA = Path("/kaggle/input/YOUR_USERNAME/zolai-bible-dataset")

# Load Zolai training corpus
import json
with open(BIBLE_DATA / "bible_zolai_combined.jsonl") as f:
    zolai_lines = [json.loads(line) for line in f]

# Load parallel pairs (Zolai ↔ English)
with open(BIBLE_DATA / "bible_zolai_english_pairs.jsonl") as f:
    pairs = [json.loads(line) for line in f]

# Load HuggingFace dataset directly
from datasets import load_from_disk
ds = load_from_disk(str(BIBLE_DATA / "hf_bible_dataset"))
print(ds["tdb77"][0])  # First Zolai verse
print(ds["parallel"][0])  # First parallel row
```

## Which Files for What

| File | Use Case |
|------|----------|
| `bible_zolai_combined.jsonl` | **Primary Zolai training corpus** — all unique verses, deduped |
| `bible_zolai_english_pairs.jsonl` | **Translation pairs** — Zolai↔English for bilingual training |
| `bible_parallel.jsonl` | **Full aligned corpus** — all 6 translations side-by-side |
| `bible_tdb77.jsonl` | Zolai-only (TDB77 edition) with verse refs |
| `bible_tedim1932.jsonl` | Zolai-only (1932 edition) — vocabulary variations |
| `bible_kjv.jsonl` | English-only (KJV) |
| `hf_bible_dataset/` | **HuggingFace format** — `load_from_disk()` ready |
| `language_patterns.json` | **Vocabulary reference** — top words, bigrams, trigrams per language |
| `per_book/` | Book-level files (132 files) — use when you need one book at a time |

## Next Pipeline Steps

### 1. Web Crawling for More Zolai Data
```python
# Use the Bible vocabulary to score and filter crawled text
with open(BIBLE_DATA / "language_patterns.json") as f:
    patterns = json.load(f)
zolai_words = set(w for w, _ in patterns["tdb77"]["top_words"])

def score_zolai(text):
    words = set(text.lower().split())
    return len(words & zolai_words) / max(len(words), 1)
```

### 2. Fine-Tune Qwen on Zolai
```python
from datasets import load_from_disk
ds = load_from_disk(str(BIBLE_DATA / "hf_bible_dataset"))
# Use ds["tdb77"] or ds["tedim1932"] for SFT training
```

### 3. Build Zolai Dictionary
```python
with open(BIBLE_DATA / "language_patterns.json") as f:
    p = json.load(f)
print(p["tdb77"]["top_words"][:50])  # Top 50 Zolai words
print(p["tdb77"]["top_bigrams"][:30])  # Common word pairs
```

### 4. Train Translation Model
```python
import json
with open(BIBLE_DATA / "bible_zolai_english_pairs.jsonl") as f:
    pairs = [json.loads(line) for line in f]
# Convert to instruction format:
for p in pairs[:3]:
    print(f"Translate to English: {p['source']}")
    print(f"Answer: {p['target']}\n")
```